# Experiment: PSO-SA Input Connectome Analysis


This notebook is intentionally thin. It rebuilds the input-side analyses from `_aPhN_DCSO_input_connectome_analysis_v1.ipynb` using shared package code instead of embedded workflow logic.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Could not locate the standalone project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from giakoumas_connectome.data import discover_project_root, load_repository_tables
from giakoumas_connectome.inputs import build_direct_input_report, export_direct_input_report
from giakoumas_connectome.plots import (
    plot_direct_input_superclass_nerve,
    plot_direct_input_superclasses,
    save_figure,
)
from giakoumas_connectome.reports import build_workflow_report

import matplotlib.pyplot as plt


## Build The Shared Report


In [ ]:
PROJECT_ROOT = discover_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'pso_sa_input_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = load_repository_tables(PROJECT_ROOT)
workflow_report = build_workflow_report('pso-sa', project_root=PROJECT_ROOT)
direct_input_report = build_direct_input_report(workflow_report, tables)
DISPLAY_LABELS = dict(zip(workflow_report.sets, workflow_report.workflow.set_labels, strict=True))

OUTPUT_DIR


## Second-Order Input Fractions


In [ ]:
summary = direct_input_report.second_order_input_percentage_summary.copy()
for column in [
    'mean_input_fraction_from_sets',
    'median_input_fraction_from_sets',
    'min_input_fraction_from_sets',
    'max_input_fraction_from_sets',
]:
    summary[column.replace('_fraction_', '_percent_')] = (summary[column] * 100).round(2)

display(
    summary[
        [
            'set_label',
            'second_order_neurons',
            'mean_input_percent_from_sets',
            'median_input_percent_from_sets',
            'min_input_percent_from_sets',
            'max_input_percent_from_sets',
        ]
    ]
)

for set_name, table in direct_input_report.second_order_input_percentages.items():
    print(DISPLAY_LABELS[set_name])
    display(table.sort_values('source_input_fraction', ascending=False).head())


## Direct Inputs To The PSO-SA Sets


In [ ]:
fig, _ = plot_direct_input_superclasses(
    direct_input_report.direct_input_superclass_counts,
    title='Superclasses of Direct Inputs to PSO-SA Sets',
)
plt.show()


In [ ]:
fig, _ = plot_direct_input_superclasses(
    direct_input_report.direct_input_superclass_counts_no_self,
    title='Direct Inputs to PSO-SA Sets Excluding Same-Set Neurons',
)
plt.show()


In [ ]:
fig, _ = plot_direct_input_superclass_nerve(
    direct_input_report.direct_input_superclass_nerve_counts_no_self,
    title='Direct Inputs to PSO-SA Sets by Superclass and Nerve',
)
plt.show()


## Exported Outputs


In [ ]:
table_paths = export_direct_input_report(direct_input_report, OUTPUT_DIR)
figure_dir = OUTPUT_DIR / 'figures'
figure_paths = {}

fig, _ = plot_direct_input_superclasses(
    direct_input_report.direct_input_superclass_counts,
    title='Superclasses of Direct Inputs to PSO-SA Sets',
)
figure_paths['direct_input_superclasses'] = save_figure(fig, figure_dir / 'direct_input_superclasses.svg')
plt.close(fig)

fig, _ = plot_direct_input_superclasses(
    direct_input_report.direct_input_superclass_counts_no_self,
    title='Direct Inputs to PSO-SA Sets Excluding Same-Set Neurons',
)
figure_paths['direct_input_superclasses_excluding_self'] = save_figure(
    fig,
    figure_dir / 'direct_input_superclasses_excluding_self.svg',
)
plt.close(fig)

fig, _ = plot_direct_input_superclass_nerve(
    direct_input_report.direct_input_superclass_nerve_counts_no_self,
    title='Direct Inputs to PSO-SA Sets by Superclass and Nerve',
)
figure_paths['direct_input_superclass_nerve_excluding_self'] = save_figure(
    fig,
    figure_dir / 'direct_input_superclass_nerve_excluding_self.svg',
)
plt.close(fig)

sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in [*table_paths.values(), *figure_paths.values()]
)
